# L91 · 注意力图解 — Self-Attention 权重热力图（Heatmap），LoRA 低秩结构可视化

**目标**：可视化 Transformer attention 的 token 关注模式，展示 LoRA 参数量节省

🔗 Aurora 连接：`aurora.llm`（KVCache、采样、TF-IDF 检索）· 视觉档案（aurora.whisper 尚未实现 — 自注意力可视化将在 aurora.multimodal 模块中）

← **上一课**　[L90 · 对话式 RAG](L90_agent.ipynb)

> 上节课学习了 **对话式 RAG**：会话记忆、来源归因与 Podcast Q&A 流水线。  
> 本课将探讨 **注意力图解**。

## 学习目标

完成本课后，你应能：

1. **手写稳定 softmax**：实现数值稳定的 `scaled_dot_product_attention`，并用 `A.sum(axis=-1) ≈ 1` 验证。
2. **解读热力图**：给定一张 `(seq, seq)` 权重图，能说出哪些 token 对相互依赖，对角线高意味着什么。
3. **计算 LoRA 参数量**：给定 `d` 和 `r`，能口算 `2·d·r` 可训练参数及节省比例 `1 - 2r/d`。

Transformer 的核心是 self-attention：每个 token 对其他 token 分配一个权重，权重越高说明这两个词在当前层的表示中越"互相依赖"。
LoRA 则是在冻结原始权重矩阵的前提下，用两个低秩矩阵 `A`（r×d）和 `B`（d×r）近似微调增量 `ΔW = B @ A`，rank r 越小，可训练参数（Trainable Parameters）越少，效率越高。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Attention 热力图

Self-attention 的输出权重矩阵 `A` 形状为 `(seq_len, seq_len)`，
元素 `A[i, j]` 表示第 `i` 个 token 对第 `j` 个 token 的关注程度。

计算流程：
```
Q = X @ W_Q   # (seq, d_k)
K = X @ W_K   # (seq, d_k)
A = softmax(Q @ K.T / sqrt(d_k))   # (seq, seq)
```

热力图中横轴是 Key（被关注的 token），纵轴是 Query（发出关注的 token）。
对角线权重高说明 token 主要关注自身；非对角线的亮点揭示语义关联。

In [ ]:
def scaled_dot_product_attention(X, W_Q, W_K, d_k):
    """手动计算 self-attention 权重矩阵"""
    Q = X @ W_Q
    K = X @ W_K
    scores = Q @ K.T / (d_k ** 0.5)
    scores_exp = np.exp(scores - scores.max(axis=-1, keepdims=True))
    A = scores_exp / scores_exp.sum(axis=-1, keepdims=True)
    return A

np.random.seed(42)
tokens = ['the', 'guitar', 'is', 'an', 'instrument']
seq_len, d_model, d_k = 5, 16, 8
X = np.random.randn(seq_len, d_model)
W_Q = np.random.randn(d_model, d_k)
W_K = np.random.randn(d_model, d_k)

A = scaled_dot_product_attention(X, W_Q, W_K, d_k)
print('Attention matrix shape:', A.shape)
print('每行之和 (应为1.0):', A.sum(axis=1).round(4))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(A, cmap='Blues', vmin=0, vmax=A.max())
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(tokens, fontsize=11)
ax.set_yticklabels(tokens, fontsize=11)
ax.set_xlabel('Key (被关注)', fontsize=11)
ax.set_ylabel('Query (发出关注)', fontsize=11)
ax.set_title('Self-Attention 热力图\n"the guitar is an instrument"', fontsize=12)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f'{A[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if A[i,j] > 0.35 else 'black')
plt.tight_layout()
plt.show()
print(f'guitar→instrument: {A[1,4]:.4f}')
print(f'guitar→guitar:     {A[1,1]:.4f}')

## 2. LoRA 参数量柱状图

LoRA 冻结原始权重 `W`（形状 `d×d`），只训练两个低秩矩阵：
```
A: (r, d)   参数量 = r * d
B: (d, r)   参数量 = d * r
ΔW = B @ A  # 等效维度仍是 (d, d)，但只有 2*d*r 个自由度
```
节省比例 = `1 - 2r/d`。当 d=4096（LLaMA-7B / GPT-3 规模）、r=8 时，节省 > 99.6%。

柱状图直观对比：不同 rank 下可训练参数 vs 原始矩阵参数。

In [ ]:
d = 4096
ranks = [1, 2, 4, 8, 16, 32, 64]
total_params = d * d
lora_params = [2 * r * d for r in ranks]
savings_pct = [(1 - lp / total_params) * 100 for lp in lora_params]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
x = range(len(ranks))
bars = ax.bar(x, lora_params, color='steelblue', alpha=0.85, label='LoRA 可训练参数')
ax.axhline(total_params, color='tomato', linewidth=2, linestyle='--',
           label=f'原始矩阵 ({total_params:,})')
ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels([f'r={r}' for r in ranks])
ax.set_ylabel('参数量（log 坐标）')
ax.set_title(f'LoRA 参数量 vs 原始矩阵\n(d={d})')
ax.legend()
for bar, lp in zip(bars, lora_params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.4,
            f'{lp:,}', ha='center', va='bottom', fontsize=8)

ax2 = axes[1]
ax2.bar(x, savings_pct, color='seagreen', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels([f'r={r}' for r in ranks])
ax2.set_ylabel('参数节省比例 (%)')
ax2.set_ylim(95, 100.5)
ax2.set_title('LoRA 节省比例 (d=4096)')
for i, sp in enumerate(savings_pct):
    ax2.text(i, sp + 0.05, f'{sp:.2f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()
print(f'r=8 节省: {savings_pct[3]:.2f}%  可训练: {lora_params[3]:,} / {total_params:,}')

## 参数实验：guitar-instrument 注意力权重（Attention Weights）

**句子**：`"the guitar is an instrument"`

**实验参数**：`exp_d_k`（注意力键维度）、`exp_seed`（随机种子）

**预期现象**：随机初始化的 W_Q / W_K 不携带语义，`guitar→instrument` 的权重
不一定高于其他 token 对。改变 `exp_d_k` 时：
- `d_k` 越小 → softmax 输入方差越大 → 分布越尖锐（某个位置权重接近1）
- `d_k` 越大 → 输入除以更大的 `sqrt(d_k)` → 分布越均匀（接近均匀分布）
在语言模型微调后的权重中，语义相关 token 对的注意力权重会显著高于无关对。

In [ ]:
# 修改 exp_d_k 或 exp_seed，观察 guitar→instrument 权重变化
exp_d_k = 4       # 试试 4 / 8 / 32 / 64
exp_seed = 0      # 改变种子观察随机波动

np.random.seed(exp_seed)
X_exp = np.random.randn(seq_len, d_model)
W_Q_exp = np.random.randn(d_model, exp_d_k)
W_K_exp = np.random.randn(d_model, exp_d_k)
A_exp = scaled_dot_product_attention(X_exp, W_Q_exp, W_K_exp, exp_d_k)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(A_exp, cmap='Blues', vmin=0, vmax=A_exp.max())
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(tokens, fontsize=11)
ax.set_yticklabels(tokens, fontsize=11)
ax.set_title(f'Attention 热力图 d_k={exp_d_k}, seed={exp_seed}', fontsize=12)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f'{A_exp[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if A_exp[i,j] > 0.35 else 'black')
plt.tight_layout()
plt.show()

guitar_idx, instrument_idx = 1, 4
print(f'd_k={exp_d_k}: guitar→instrument = {A_exp[guitar_idx, instrument_idx]:.4f}')
print(f'd_k={exp_d_k}: guitar→guitar     = {A_exp[guitar_idx, guitar_idx]:.4f}')
print('提示：d_k 越大 softmax 分布越均匀，越小分布越尖锐')

## 本课收束

`scaled_dot_product_attention` 输出的 `(seq, seq)` 权重矩阵，直接可视化为热力图，揭示每层 token 之间的注意力分配模式。
LoRA 的参数节省来自 `ΔW = B @ A` 的低秩分解（Low-Rank Decomposition），rank r 决定可训练参数量 `2·d·r`，这两个核心量都将进入 `aurora.llm` 的微调模块。
本模块 LLM 应用告一段落。

---

→ **下一课**　[L92 · 端到端流水线](../10_integration/L92_pipeline.ipynb)

> 下节课将学习 **端到端流水线**：麦克风 → ASR → LLM → 文本回答，全链路组装。